# Cruzamento Análise de Performance das Aulas Semanais na BF25
### Objetivo: Nesse pipeline pretendemos analisar quantos leads inscritos na campanha Isca-aulassemanais se inscreveram também na BF25. Além disso queremos responder:

#### a. Quantos desses leads vieram pelo Whatsapp;
#### b. Quantos disparos esses leads receberam ao total;
#### c. Qual o percentual de compra desses leads por Origem;

In [1]:
# Importações básicas
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Adiciona src ao path
sys.path.append('../src')

#PRODUCT Ids
CRONOGRAMA_ID = '3310674'

#Subdomains do Hotmart Club
CRONOGRAMA_SUBDOMAIN = 'cronogramadosfluentes-xwamel'

#Aulas de Conversação 
AULAS_DE_CONVERSACAO_SUBDOMAIN = 'aulasdeconversacao-vpqdjd'

#Ingles Express Travel
INGLES_EXPRESS_TRAVEL_SUBDOMAIN = 'inglesexpresstravel'

#Intensivão do Inglês
INTENSIVAO_DO_INGLES_BASICO_SUBDOMAIN = 'intensivaodoinglesbasico'

#Aulas de Conversação Particulares
AULAS_DE_CONVERSACAO_PARTICULARES_SUBDOMAIN = 'aulasdeconversacao-avjbpp'

# Utilitários de dados
from data_utils import (
    load_raw_data,
    save_processed_data,
    remove_duplicates,
    handle_missing_values,
    detect_outliers,
    normalize_column,
    process_phone_string,
    process_phone_number,
    clean_and_lower_column,
    flatten_list_to_df, 
    timestamp_ms_to_datetime
)

# Utilitários SQL
from sql_utils import DatabaseConnection as Dbc

# Utilitários de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitários de API
from api_utils import (
    make_request,
    get_json,
    post_json,
    paginated_request,
    response_to_dataframe
)

# utilitários hotmart
from hotmart_utils import Hotmart

# utilitários mongo
from mongo_utils import MongoDBConnection

# utilitários tmb
from tmb_utils import TMB   

# Configurações pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load Database Driver
db = Dbc()

# Inicializar API Hotmart
hotmart = Hotmart()

# Inicializar API TMB
tmb = TMB()

print('✓ Importações concluídas com sucesso!')

✓ Importações concluídas com sucesso!


In [2]:
with MongoDBConnection(database_name="disparos_api") as mongo:
    # Buscar apenas documentos com status EXECUTED
    results = mongo.get_documents_from_filtered_collections(
        collection_filter="lcto-bf-nov/25_w_c"
    )


✅ Conectado ao MongoDB - Database: disparos_api
📋 Encontradas 76 coleções contendo 'lcto-bf-nov/25_w_c'

🔍 Buscando documentos em 76 coleções...

📄 8723 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c41'
📄 2700 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c23-correto'
📄 451 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c4'
📄 8724 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c38'
📄 451 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c3'
📄 3001 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c_u8'
📄 7751 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c23'
📄 2474 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c11_v1-mkt2'
📄 3424 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c11_v1-3420'
📄 550 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c_a4-2envio-500'
📄 451 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c1'
📄 5000 documentos encontrados na coleção 'mkt_lcto-bf-nov/25_w_c_u14-5000'
📄

In [3]:

"""
Cell generated by Data Wrangler.
"""
def clean_data(results):
    results['phone'] = process_phone_string(results, 'phone')
    # Performed 1 aggregation grouped on column: 'phone'
    results = results.groupby(['phone']).agg(_id_nunique=('_id', 'nunique')).reset_index()
    return results

results_clean = clean_data(results.copy())
results_clean.head()

,phone,_id_nunique
0,0,1
1,000,2
2,00000000,2
3,00000000000,2
4,000000000000,1


# 1. Importando leads Inscritos na isca-aulassemanais

In [4]:
df = db.execute_query_from_file('select_compleete_lead_data_from_campaign', params={'conversion_type_ids': 1, 'campaign_id': 'isca-aulassemanais'})

# 2. Importando leads Inscritos na BF25
df_bf25 = db.execute_query_from_file('select_compleete_lead_data_from_campaign', params={'conversion_type_ids': 1, 'campaign_id': 'lcto-bf25-leads'})

## 1.1 Buscando e tratando dados de Venda na Black Friday

In [5]:
df_hotmart = load_raw_data('hotmart.csv', sep=';')

def clean_data(hotmart_transactions):
    # Drop columns: 'Nome do Produtor', 'Documento do Produtor' and 49 other columns
    hotmart_transactions = hotmart_transactions.drop(columns=['Nome do Produtor', 'Documento do Produtor', 'Nome do Afiliado', 'Transação', 'Meio de Pagamento', 'Origem', 'Moeda', 'Preço do Produto', 'Moeda.1', 'Preço da Oferta', 'Taxa de Câmbio', 'Moeda.2', 'Preço Original', 'Número da Parcela', 'Data de Venda', 'Data de Confirmação', 'Status', 'Documento', 'CEP', 'Cidade', 'Estado', 'Bairro', 'País', 'Endereço', 'Número', 'Complemento', 'chave', 'Código do Produto', 'Código da Afiliação', 'Tipo de Pagamento', 'Período Grátis', 'Tem co-produção', 'Venda feita como', 'Preço Total', 'Tipo pagamento oferta', 'Taxa de Câmbio Real', 'Preço Total Convertido', 'Quantidade de itens', 'Oferta de Upgrade', 'Cupom', 'Moeda.3', 'Valor que você recebeu convertido', 'Data Vencimento', 'Taxa de Câmbio do valor recebido', 'Instagram', 'Origem da venda', 'Moeda de recebimento', 'Faturamento líquido', 'Código do assinante', 'Nota Fiscal', 'Valor do frete bruto'])
    # Filter rows based on column: 'Recorrência'
    hotmart_transactions = hotmart_transactions[(hotmart_transactions['Recorrência'] == 1) | (hotmart_transactions['Recorrência'].isna())]
    hotmart_transactions['DDD'] = process_phone_string(hotmart_transactions, 'DDD')
    hotmart_transactions['Telefone'] = process_phone_string(hotmart_transactions, 'Telefone')
    hotmart_transactions['phone'] = hotmart_transactions.apply(lambda x: x['Telefone'] if pd.isna(x['DDD']) else f'55{x["DDD"]}{x["Telefone"]}', axis=1)
    # Drop column: 'DDD'
    hotmart_transactions = hotmart_transactions.drop(columns=['DDD', 'Telefone'])

    # Drop column: 'Recorrência'
    hotmart_transactions = hotmart_transactions.drop(columns=['Recorrência'])
    # Rename column 'Nome do Produto' to 'product_name'
    hotmart_transactions = hotmart_transactions.rename(columns={'Nome do Produto': 'product_name', "Nome": "name", "Email": "email","Código de Oferta": "offer_id" })
    # Drop column: 'Origem de Checkout'
    hotmart_transactions = hotmart_transactions.drop(columns=['Origem de Checkout'])
    hotmart_transactions = hotmart_transactions[hotmart_transactions['product_name'] == 'Cronograma dos Fluentes ']
    return hotmart_transactions

df_hotmart = clean_data(df_hotmart.copy())
df_hotmart

,product_name,name,email,offer_id,phone
0,Cronograma dos Fluentes,Lilian Cristina Dornelles,lilian.dornelles@hotmail.com,3pq6l619,5541999312716
1,Cronograma dos Fluentes,CARLA CRISTINA CAPELLETO ZULIANI,ccapelleto@yahoo.com.br,rs1vpp7f,5512981216059
6,Cronograma dos Fluentes,Marina Maluta,marinaqueirozmaluta@gmail.com,pj5p39qx,5511953617514
7,Cronograma dos Fluentes,Alessandra Lenzi,ale.lenzi.campos@gmail.com,pj5p39qx,5511975023435
8,Cronograma dos Fluentes,Diogo Pinheiro Sampaio,dsampaio13@gmail.com,8pilk3ej,5577998661313
...,...,...,...,...,...
3396,Cronograma dos Fluentes,Eduardo Ferreira Santana,eduardof.santana@outlook.com,igxwngei,5511967016960
3397,Cronograma dos Fluentes,ALANA AUGUSTA LISBOA,alana@atuarassessoria.com.br,igxwngei,5531985154623
3398,Cronograma dos Fluentes,Thiago julio,thiagojulio20@gmail.com,igxwngei,5548998551473
3399,Cronograma dos Fluentes,Marcelo Araújo e Silva,marceloas78@gmail.com,igxwngei,5561981861510


In [6]:
df_tmb = load_raw_data('tmb.csv', sep=';')

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_tmb):
    # Drop columns: 'Produtor', 'Pedido' and 29 other columns
    df_tmb = df_tmb.drop(columns=['Produtor', 'Pedido', 'Endereço completo', 'Logradouro', 'Número', 'Bairro', 'Complemento', 'CEP', 'Estado', 'Cidade', 'País', 'Status', 'Status Financeiro', 'Status Cancelamento', 'Ticket (R$)', 'Modalidade de Contrato', 'Criado Em', 'Data Efetivado', 'Data Cancelado', 'utm_source', 'utm_medium', 'utm_campaign', 'utm_content', 'utm_last_source', 'utm_last_medium', 'utm_last_campaign', 'utm_last_content', 'Renovação automatica', 'Data de Renovação Prevista', 'Renovação enviada', 'Oferta'])
    # Drop column: 'Cliente CPF'
    df_tmb = df_tmb.drop(columns=['Cliente CPF'])
    # Rename column 'Produto' to 'offer_code'
    df_tmb = df_tmb.rename(columns={'Produto': 'offer_id', "Cliente Nome": "name", "Cliente Email": "email", "Telefone": "phone"})

    df_tmb['phone'] = process_phone_string(df_tmb, 'phone')
    
    return df_tmb

df_tmb = clean_data(df_tmb.copy())
df_tmb.head()

,offer_id,name,email,phone
0,COM-BF25-Oferta Negociação,William Fernando Lopes Rolim Nunes,wescorpiaum@gmail.com,5515996907746
1,COM-BF25-Oferta Pós-Live,Daffiny Damiany faria de Nogueira,daffynny@hotmail.com,5565992971784
2,COM-BF25-Oferta Negociação,NELSON HENRIQUE FERREIRA POSSIDIO,nelsonpossidio@gmail.com,5571996655731
3,COM-BF25-alunos-Oferta Negociação,Vanessa Fernandes,vanessamailbox88@gmail.com,5532991565691
4,COM-BF25-alunos-Oferta Live,Davi Lucas Amaral da Silva,davilucasstudy@gmail.com,5521966360773


In [7]:
df_buyers = pd.concat([df_hotmart, df_tmb], ignore_index=True)
df_buyers.head()

,product_name,name,email,offer_id,phone
0,Cronograma dos Fluentes,Lilian Cristina Dornelles,lilian.dornelles@hotmail.com,3pq6l619,5541999312716
1,Cronograma dos Fluentes,CARLA CRISTINA CAPELLETO ZULIANI,ccapelleto@yahoo.com.br,rs1vpp7f,5512981216059
2,Cronograma dos Fluentes,Marina Maluta,marinaqueirozmaluta@gmail.com,pj5p39qx,5511953617514
3,Cronograma dos Fluentes,Alessandra Lenzi,ale.lenzi.campos@gmail.com,pj5p39qx,5511975023435
4,Cronograma dos Fluentes,Diogo Pinheiro Sampaio,dsampaio13@gmail.com,8pilk3ej,5577998661313


In [8]:
df_offers = load_raw_data('offers.csv')

"""
Cell generated by Data Wrangler.
"""
def clean_data(df_offers):
    # Drop columns: 'updated_at', 'created_at' and 13 other columns
    df_offers = df_offers.drop(columns=['updated_at', 'created_at', 'status', 'description', 'start_date', 'end_date', 'price', 'main_product_id', 'access_duration_months', 'is_default', 'entry_value', 'installment_12x_value', 'is_child', 'comercial_use', 'price_type'])
    # Filter rows based on column: 'campaign_id'
    df_offers = df_offers[df_offers['campaign_id'].str.contains("lcto-bf25", regex=False, na=False, case=False)]
    return df_offers

df_offers = clean_data(df_offers.copy())
df_offers.head()

,offer_id,name,campaign_id,platform,base_url
7,COM-BF25-alunos-Oferta Downsell,[TMB][BF25A] Oferta Downsell\t,lcto-bf25-alunos,TMB,https://pay.tmbeducacao.com.br/MatheusWerner/Z...
12,aflr0gju,[BF25] Oferta Live - v2,lcto-bf25-leads,Hotmart,https://pay.hotmart.com/M86482747L?off=aflr0gju
15,i2jmb51w,[COM][BF25A] Oferta Pós-Live 2 - Filha,lcto-bf25-alunos,Hotmart,https://pay.hotmart.com/M86482747L?off= i2jmb51w
23,7ne4llz3,[COM][BF25] Oferta Pós-Live,lcto-bf25-leads,Hotmart,https://pay.hotmart.com/M86482747L?off=7ne4llz...
24,selbsff3,[COM] [BF25] Upsell Oferta 1,lcto-bf25-alunos,Hotmart,https://pay.hotmart.com/Q92997304O?off=selbsff...


In [9]:
df_buyers_offers = pd.merge(df_buyers, df_offers, on='offer_id', how='left')
df_buyers_offers = df_buyers_offers.dropna(subset=['campaign_id'])
df_buyers_offers = df_buyers_offers[df_buyers_offers['campaign_id'].str.contains("lcto-bf25-leads", regex=False, na=False, case=False)]

## 1.1. Cruzando dados de inscrição

### Protocolo: 1) Cruzamento por lead_id, 2) Cruzamento por email, 3) Cruzamento por phone

In [10]:
# 1. Cruzar por lead_id
merged_leads = pd.merge(df, df_bf25, on='lead_id', how='inner', suffixes=('_aulas', '_bf25'))

# 2. Cruzar por email (ignorando os leads já encontrados pelo lead_id)
df_not_found_lead_id = df[~df['lead_id'].isin(merged_leads['lead_id'])]
df_bf25_not_found_lead_id = df_bf25[~df_bf25['lead_id'].isin(merged_leads['lead_id'])]

df_not_null_email = df_not_found_lead_id[df_not_found_lead_id['email'].notnull()]
df_bf25_not_null_email = df_bf25_not_found_lead_id[df_bf25_not_found_lead_id['email'].notnull()]

merged_email = pd.merge(df_not_null_email, df_bf25_not_null_email, on='email', how='inner', suffixes=('_aulas', '_bf25'))

# 3. Cruzar por phone (ignorando os leads já encontrados por lead_id e email)
leads_found_ids = set(merged_leads['lead_id']).union(set(merged_email.get('lead_id_aulas', [])), set(merged_email.get('lead_id_bf25', [])))

df_not_found_email = df[~df['lead_id'].isin(leads_found_ids) & df['phone'].notnull()]
df_bf25_not_found_email = df_bf25[~df_bf25['lead_id'].isin(leads_found_ids) & df_bf25['phone'].notnull()]

merged_phone = pd.merge(df_not_found_email, df_bf25_not_found_email, on='phone', how='inner', suffixes=('_aulas', '_bf25'))

# Concatenar todos os cruzamentos
df_cruzado = pd.concat([merged_leads, merged_email, merged_phone], ignore_index=True)

df_cruzado['phone_aulas'] = process_phone_string(df_cruzado, 'phone_aulas')
df_cruzado['phone_bf25'] = process_phone_string(df_cruzado, 'phone_bf25')

df_cruzado['phone'] = df_cruzado.apply(lambda x: x['phone_aulas'] if pd.notna(x['phone_aulas']) else x['phone_bf25'], axis=1)


df_cruzado['email_aulas'] = df_cruzado['email_aulas'].str.lower().str.strip()
df_cruzado['email_bf25'] = df_cruzado['email_bf25'].str.lower().str.strip()

df_cruzado['email'] = df_cruzado.apply(lambda x: x['email_aulas'] if pd.notna(x['email_aulas']) else x['email_bf25'], axis=1)

df_cruzado = df_cruzado.drop(columns=['phone_aulas', 'phone_bf25', 'email_aulas', 'email_bf25'])


In [11]:
results_clean = results_clean[results_clean['phone'].isin(df_cruzado['phone'].unique())]

## 1.2. Cruzando dados de Venda



In [12]:
# Assumindo que os dados de vendas estão em um DataFrame chamado df_offer_buyers,
# e que ele possui colunas 'email' e 'phone'

# Padronizar email para facilitar o merge
df_cruzado['email'] = df_cruzado['email'].str.lower().str.strip()
df_buyers_offers['email'] = df_buyers_offers['email'].str.lower().str.strip()

# Padronizar phone para facilitar o merge
df_cruzado['phone'] = df_cruzado['phone'].astype(str).str.replace(r'\D', '', regex=True).str.strip()
df_buyers_offers['phone'] = df_buyers_offers['phone'].astype(str).str.replace(r'\D', '', regex=True).str.strip()

# Marca quem comprou por email
cruzou_email = df_cruzado['email'].isin(df_buyers_offers['email'])

# Marca quem comprou por phone (ignorando quem já foi achado por email)
emails_cruzados = df_cruzado.loc[cruzou_email, 'email']
phones_nao_cruzados_email = df_cruzado.loc[~cruzou_email, 'phone']

cruzou_phone = phones_nao_cruzados_email.isin(df_buyers_offers['phone'])

# Cria a coluna 'comprou'
df_cruzado['comprou'] = False
df_cruzado.loc[cruzou_email, 'comprou'] = True
df_cruzado.loc[~cruzou_email, 'comprou'] = df_cruzado.loc[~cruzou_email, 'phone'].isin(df_buyers_offers['phone'])

In [18]:
# Tabela de resumo de leads e vendas por UTM Source (BF25)

# --- Agregação dos dados (caso não tenha sido feito no notebook)
agregado_leads = df_cruzado.groupby('utm_source_bf25').agg(
    leads_unicos=('lead_id', 'nunique'),
    leads_compraram=('comprou', lambda x: df_cruzado.loc[x.index, 'lead_id'][x].nunique())
).reset_index()

# Calcular total geral de leads para o percentual de participação
total_leads = agregado_leads['leads_unicos'].sum()

agregado_leads['% do total de leads'] = (agregado_leads['leads_unicos'] / total_leads * 100).round(2)
agregado_leads['% conversão em vendas'] = (agregado_leads['leads_compraram'] / agregado_leads['leads_unicos'] * 100).round(2)

# Reordenar e renomear colunas para visualização mais amigável
tabela_resumo = agregado_leads.rename(columns={
    'utm_source_bf25': 'Origem (utm_source_bf25)',
    'leads_unicos': 'Quantidade de Leads',
    'leads_compraram': 'Quantidade de Vendas'
})[
    [
        'Origem (utm_source_bf25)', 
        'Quantidade de Leads', 
        'Quantidade de Vendas', 
        '% do total de leads', 
        '% conversão em vendas'
    ]
].sort_values(by='Quantidade de Leads', ascending=False).reset_index(drop=True)

# Exibir a tabela formatada no notebook
import pandas as pd
from IPython.display import display

display(tabela_resumo.style
    .format({
        'Quantidade de Leads': '{:,.0f}',
        'Quantidade de Vendas': '{:,.0f}',
        '% do total de leads': '{:,.2f}%',
        '% conversão em vendas': '{:,.2f}%'
    })
    .set_caption('Resumo de Leads, Vendas e Conversão por Origem (utm_source_bf25)')
    .set_properties(**{'text-align': 'center'})
)


,Origem (utm_source_bf25),Quantidade de Leads,Quantidade de Vendas,% do total de leads,% conversão em vendas
0,whatsapp,"1,687",69,49.24%,4.09%
1,meta-ads,778,42,22.71%,5.40%
2,instagram,491,52,14.33%,10.59%
3,youtube,327,52,9.54%,15.90%
4,email,60,11,1.75%,18.33%
5,tiktok,34,3,0.99%,8.82%
6,fb,12,1,0.35%,8.33%
7,ig,11,2,0.32%,18.18%
8,links.matheusjerke.com.br,9,0,0.26%,0.00%
9,direto,6,0,0.18%,0.00%


In [29]:
leads_aulas_semanais_total = 20112
leads_capturados = len(df_cruzado['lead_id'].unique())
vendas_realizadas =  180 #len(df_cruzado[df_cruzado['comprou'] == True]['lead_id'].unique())
faturamento_bruto_aproximado = vendas_realizadas * 2398.80
faturamento_liquido_aproximado = vendas_realizadas * 1329.76

custo_total_de_disparo = leads_aulas_semanais_total * 4 * 0.04

cpl = custo_total_de_disparo / leads_capturados
roi = faturamento_liquido_aproximado / custo_total_de_disparo

print(f"Total de leads da campanha Isca-AulasSemanais: {leads_aulas_semanais_total:,}")
print(f"Total de leads capturados: {leads_capturados:,}")
print(f"Custo total de disparo: R$ {custo_total_de_disparo:,.2f}")
print(f"Custo por lead capturado: R$ {cpl:,.2f}")

print(f"Vendas realizadas: {vendas_realizadas:,}")
print(f'CONVERSÃO EM VENDAS: {vendas_realizadas / leads_capturados * 100:,.2f}%')
print(f"Faturamento bruto aproximado: R$ {faturamento_bruto_aproximado:,.2f}")
print(f"Faturamento líquido aproximado: R$ {faturamento_liquido_aproximado:,.2f}")
print(f"ROI: {roi:,.2f}")


Total de leads da campanha Isca-AulasSemanais: 20,112
Total de leads capturados: 3,280
Custo total de disparo: R$ 3,217.92
Custo por lead capturado: R$ 0.98
Vendas realizadas: 180
CONVERSÃO EM VENDAS: 5.49%
Faturamento bruto aproximado: R$ 431,784.00
Faturamento líquido aproximado: R$ 239,356.80
ROI: 74.38
